In [1]:
import os
import csv
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from itertools import combinations
from collections import defaultdict, Counter
from scipy import stats

In [ ]:
for (a, b, c) in [()]

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
Data_Root = '/kellogg/proj/hpk5609/Patent/'

In [4]:
def yield_one_line(filename, delimiter=',', quoting = csv.QUOTE_ALL):
    '''a generator which produce one line of a given file'''
    with open(filename, 'r') as file:
        reader = csv.reader(file, delimiter=delimiter, quoting=quoting)
        count = 0
        for row in reader:
            count += 1
            if count % 1000000 == 0:
                print('processed %d lines...' % (count))
            yield row

In [5]:
pid_year = {}
pid_appid = {}

for line in yield_one_line(Data_Root+'pg_published_application.tsv', delimiter='\t', quoting=csv.QUOTE_ALL):
    if line[0] != 'pgpub_id':
        pid, appid, filing_date, patent_type, title, abst = line[0], line[1], line[2], line[3], line[8], line[9]
        if patent_type == 'utility' and filing_date != '':
            pid, year = int(pid), int(filing_date[:4])
            pid_year[pid] = year
            pid_appid[pid] = appid

processed 1000000 lines...
processed 2000000 lines...
processed 3000000 lines...
processed 4000000 lines...
processed 5000000 lines...
processed 6000000 lines...
processed 7000000 lines...


In [6]:
len(pid_year)

6810940

In [7]:
len(pid_appid)

6810940

In [27]:
sorted(Counter(pid_year.values()).items(), key = lambda x: x[0])

[(1909, 1),
 (1911, 1),
 (1913, 1),
 (1918, 3),
 (1919, 2),
 (1991, 1),
 (1994, 2),
 (1995, 5),
 (1996, 7),
 (1997, 5),
 (1998, 4),
 (1999, 19),
 (2000, 137),
 (2001, 2543),
 (2002, 14519),
 (2003, 90406),
 (2004, 232572),
 (2005, 307803),
 (2006, 323549),
 (2007, 327731),
 (2008, 316441),
 (2009, 295917),
 (2010, 307739),
 (2011, 331987),
 (2012, 353861),
 (2013, 373574),
 (2014, 380386),
 (2015, 379779),
 (2016, 380895),
 (2017, 389395),
 (2018, 392372),
 (2019, 416932),
 (2020, 407676),
 (2021, 382310),
 (2022, 279430),
 (2023, 122935)]

|P| (yearly cumulative)

In [87]:
years = list(range(2001, 2024))

In [101]:
tem = Counter(pid_year.values())
n_year_count = np.cumsum([tem[y] if y in tem else 0 for y in years])
# n_year_count = dict(zip(years, n_year_count))

In [102]:
# from 2001 to 2023 (inclusive)
n_year_count

array([   2543,   17062,  107468,  340040,  647843,  971392, 1299123,
       1615564, 1911481, 2219220, 2551207, 2905068, 3278642, 3659028,
       4038807, 4419702, 4809097, 5201469, 5618401, 6026077, 6408387,
       6687817, 6810752])

In [181]:
pid_class = defaultdict(list)
pid_subclass = defaultdict(list)
pid_group = defaultdict(list)
pid_first_code = {}

In [184]:
for line in yield_one_line(Data_Root+'pg_cpc_at_issue.tsv', delimiter='\t', quoting=csv.QUOTE_ALL):
    if line[0] != 'pgpub_id':
        pid, seq, cpc_class, cpc_subclass, cpc_group = line[0], line[1], line[4], line[5], line[6]
        pid, seq = int(pid), int(seq)
        pid_class[pid].append(cpc_class)
        pid_subclass[pid].append(cpc_subclass)
        pid_group[pid].append(cpc_group)
        if seq == 1:
            pid_first_code[pid] = cpc_group

processed 1000000 lines...
processed 2000000 lines...
processed 3000000 lines...
processed 4000000 lines...
processed 5000000 lines...
processed 6000000 lines...
processed 7000000 lines...
processed 8000000 lines...
processed 9000000 lines...
processed 10000000 lines...
processed 11000000 lines...
processed 12000000 lines...
processed 13000000 lines...
processed 14000000 lines...
processed 15000000 lines...
processed 16000000 lines...
processed 17000000 lines...
processed 18000000 lines...


In [17]:
pid_class[20220221953]

['H01', 'H01', 'H01', 'G06', 'G06', 'G06', 'G06', 'G06', 'H01']

In [18]:
pid_subclass[20220221953]

['H01L', 'H01L', 'H01L', 'G06F', 'G06F', 'G06F', 'G06F', 'G06F', 'H01L']

In [19]:
pid_group[20220221953]

['H01L27/323',
 'H01L27/3234',
 'H01L27/3244',
 'G06F3/04166',
 'G06F3/0412',
 'G06F3/0443',
 'G06F3/0446',
 'G06F3/0445',
 'H01L51/5281']

In [29]:
# 6.8 - 4.1 = 2.7M apps have no CPC information
len(pid_group)

4113850

In [174]:
cpc_num_count = defaultdict(int)

for pid in pid_group:
    # utility patents in year [2001, 2023]
    if pid in pid_year:
        year = pid_year[pid]
        if year >= 2001 and year <= 2023:
            cpc_num_count[len(pid_group[pid])] += 1

In [177]:
sorted(cpc_num_count.items())

[(1, 629990),
 (2, 665616),
 (3, 672142),
 (4, 579130),
 (5, 437796),
 (6, 311464),
 (7, 216370),
 (8, 152378),
 (9, 108576),
 (10, 78630),
 (11, 58213),
 (12, 43626),
 (13, 33023),
 (14, 24850),
 (15, 19451),
 (16, 14695),
 (17, 11793),
 (18, 9269),
 (19, 7447),
 (20, 6018),
 (21, 4772),
 (22, 3986),
 (23, 3314),
 (24, 2753),
 (25, 2256),
 (26, 1980),
 (27, 1546),
 (28, 1374),
 (29, 1149),
 (30, 1076),
 (31, 932),
 (32, 724),
 (33, 656),
 (34, 625),
 (35, 532),
 (36, 496),
 (37, 350),
 (38, 365),
 (39, 321),
 (40, 322),
 (41, 241),
 (42, 283),
 (43, 255),
 (44, 188),
 (45, 240),
 (46, 170),
 (47, 151),
 (48, 113),
 (49, 160),
 (50, 109),
 (51, 92),
 (52, 112),
 (53, 84),
 (54, 67),
 (55, 72),
 (56, 73),
 (57, 58),
 (58, 56),
 (59, 46),
 (60, 73),
 (61, 66),
 (62, 34),
 (63, 25),
 (64, 41),
 (65, 35),
 (66, 32),
 (67, 28),
 (68, 27),
 (69, 18),
 (70, 17),
 (71, 47),
 (72, 16),
 (73, 61),
 (74, 17),
 (75, 29),
 (76, 15),
 (77, 31),
 (78, 23),
 (79, 17),
 (80, 9),
 (81, 8),
 (82, 14),
 (

Count of individual cpc code (yearly cumulative)

In [103]:
obs_code_year_count = defaultdict(lambda: defaultdict(int))

In [104]:
for pid in pid_group:
    codes = pid_group[pid]
    # utility patents in year [2001, 2023] with at least 2 cpc codes
    if pid in pid_year and len(codes) >= 2:
        year = pid_year[pid]
        if year >= 2001 and year <= 2023:
            for code in codes:
                obs_code_year_count[code][year] += 1

In [105]:
obs_code_year_count['G06F3/0443']

defaultdict(int,
            {2021: 277,
             2017: 20,
             2019: 132,
             2020: 226,
             2023: 122,
             2022: 207,
             2018: 51,
             2016: 1})

In [106]:
for code in obs_code_year_count.keys():
    tt = obs_code_year_count[code]
    cn_each_year = [tt[y] if y in tt else 0 for y in years]
    cn_cum_year = np.cumsum(cn_each_year)
    obs_code_year_count[code] = cn_cum_year

In [107]:
obs_code_year_count['G06F3/0443']

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    1,   21,   72,  204,  430,  707,  914,
       1036])

In [147]:
obs_code_year_count['G06F3/0446']

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    1,    5,   47,  135,  424,  904, 1548, 2100,
       2394])

In [108]:
len(obs_code_year_count)

219860

Count of cpc code pair (yearly cumulative)

In [117]:
obs_pair_year_count = defaultdict(lambda: defaultdict(int))

for pid in pid_group:
    codes = sorted(pid_group[pid])
    # utility patents in year [2001, 2023] with at least 2 cpc codes
    if pid in pid_year and len(codes) >= 2:
        year = pid_year[pid]
        if year >= 2001 and year <= 2023:
            for c_pair in combinations(codes, 2):
                obs_pair_year_count[c_pair][year] += 1

In [119]:
obs_pair_year_count[('G06F3/0443', 'G06F3/0446')]

defaultdict(int,
            {2021: 147,
             2017: 9,
             2020: 128,
             2023: 76,
             2022: 111,
             2018: 15,
             2019: 49})

In [120]:
# accumulate counts and order by year

for c_pair in list(obs_pair_year_count.keys()):
    tt = obs_pair_year_count[c_pair]
    cn_each_year = [tt[y] if y in tt else 0 for y in years]
    cn_cum_year = np.cumsum(cn_each_year)
    obs_pair_year_count[c_pair] = cn_cum_year

In [121]:
obs_pair_year_count[('G06F3/0443', 'G06F3/0446')]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   9,  24,  73, 201, 348, 459, 535])

In [118]:
len(obs_pair_year_count)

17926548

Expected count and variance of cpc code pair (analytical solution)

In [122]:
exp_pair_year_count = {}
exp_pair_year_var = {}

for c_pair in obs_pair_year_count:
    ca, cb = c_pair
    na, nb = obs_code_year_count[ca], obs_code_year_count[cb]
    nsum = n_year_count
    exp_n = na * nb / nsum
    exp_pair_year_count[c_pair] = exp_n
    # var
    var = exp_n * (1 - na/nsum) * (nsum - nb) / (nsum - 1)
    exp_pair_year_var[c_pair] = var

In [127]:
exp_pair_year_count[('G06F3/0443', 'G06F3/0446')]

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.13129799e-06,
       2.05236035e-04, 1.86870286e-03, 1.53951275e-02, 6.45063115e-02,
       1.70781821e-01, 2.86999480e-01, 3.64157144e-01])

In [128]:
exp_pair_year_var[('G06F3/0443', 'G06F3/0446')]

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.13129671e-06,
       2.05233176e-04, 1.86862885e-03, 1.53934095e-02, 6.44920430e-02,
       1.70721757e-01, 2.86870193e-01, 3.63973822e-01])

In [123]:
len(exp_pair_year_count)

17926548

In [124]:
len(exp_pair_year_var)

17926548

Aggregate code to class

In [137]:
obs_count = defaultdict(lambda: np.array([0] * 23))
exp_count = defaultdict(lambda: np.array([0.0] * 23))
exp_var = defaultdict(lambda: np.array([0.0] * 23))

for c_pair in obs_pair_year_count:
    ca, cb = c_pair
    class_a, class_b = ca[:3], cb[:3]
    obs_count[(class_a, class_b)] += obs_pair_year_count[c_pair]
    exp_count[(class_a, class_b)] += exp_pair_year_count[c_pair]
    exp_var[(class_a, class_b)] += exp_pair_year_var[c_pair]

In [140]:
for c_pair in obs_count:
    class_a, class_b = c_pair
    if class_a == class_b:
        obs_count[c_pair] = obs_count[c_pair] / 2
        exp_count[c_pair] = exp_count[c_pair] / 2
        exp_var[c_pair] = exp_var[c_pair] / 2

In [163]:
zscores = {}
for c_pair in obs_count:    
    zs_li = (obs_count[c_pair] - exp_count[c_pair]) / np.sqrt(exp_var[c_pair])
    zscores[c_pair] = dict(zip(years, zs_li))

/tmp/ipykernel_53720/2260490129.py:3: RuntimeWarning: invalid value encountered in true_divide
  zs_li = (obs_count[c_pair] - exp_count[c_pair]) / np.sqrt(exp_var[c_pair])


In [164]:
zscores[('G06', 'G06')]

{2001: nan,
 2002: 182.48971220691504,
 2003: 801.7862009236228,
 2004: 1093.4358363738068,
 2005: 1264.7881007893661,
 2006: 1417.689357973215,
 2007: 1643.6921396541363,
 2008: 1919.5097181371616,
 2009: 2013.187445568423,
 2010: 2016.2995709841123,
 2011: 2110.5254854335444,
 2012: 2552.6842681386993,
 2013: 2965.9965018921903,
 2014: 3679.695176730822,
 2015: 4441.231677666478,
 2016: 5002.1513099576805,
 2017: 5659.0798714899365,
 2018: 6320.985891635072,
 2019: 7004.35348335009,
 2020: 7640.790344000909,
 2021: 8030.928798747406,
 2022: 8409.82189848086,
 2023: 8648.800629222114}

In [159]:
len(zscores)

6944

In [166]:
with open(Data_Root+'class_pair_novelty.json', 'w') as ofile:
    for c_pair, info in zscores.items():
        class_a, class_b = c_pair
        row = {'pair': class_a + '-' + class_b, 'scores': info}
        ofile.write(json.dumps(row) + '\n')

In [167]:
zscores = {}
with open(Data_Root+'class_pair_novelty.json', 'r') as ofile:
    for row in ofile:
        row = json.loads(row)
        c_pair, info = row['pair'], row['scores']
        class_a, class_b = c_pair.split('-')
        info_bar = {}
        for year, sc in info.items():
            info_bar[int(year)] = sc
        zscores[(class_a, class_b)] = info_bar

Calculate novelty score for each application

In [171]:
pid_scores = defaultdict(list)

for pid in pid_class:
    codes = sorted(pid_class[pid])
    # utility patents in year [2001, 2023] with at least 2 cpc codes
    if pid in pid_year and len(codes) >= 2:
        year = pid_year[pid]
        if year >= 2001 and year <= 2023:
            for c_pair in combinations(codes, 2):
                pid_scores[pid].append(zscores[c_pair][year])

In [185]:
len(pid_scores)

3483553

In [186]:
with open(Data_Root+'utility_app_2001_2023_cpc_novelty.json', 'w') as ofile:
    for pid in pid_scores:
        row = {'pgpub_id': pid, 'application_id': pid_appid[pid], 'year': pid_year[pid], 'first_cpc_group': pid_first_code[pid], 'cpc_group': pid_group[pid], 'scores': pid_scores[pid]}
        ofile.write(json.dumps(row) + '\n')